<a href="https://colab.research.google.com/github/YefridC09/ST-554-Project1-Template/blob/main/Task3/Task_3_Project1_ST_554.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3 - Project 1

## Author: Yefrid Cordoba

### Importing data and modules

First install the repo where the data is stored

In [18]:
!pip install ucimlrepo

From the repo, we extract the list of air quality for our analysis, including filtering not existing data for C6H6(GT), CO(GT), T, RH, or AH

In [19]:
import ucimlrepo as uci
import numpy as np
import pandas as pd
import sklearn as sk


air_quality = uci.fetch_ucirepo(id=360)
air_quality = air_quality.data.features
air_quality = air_quality[["Date", "Time", "C6H6(GT)",
                           "CO(GT)", "T", "RH", "AH"]]
#print(air_quality.info())
#Filtering the data for missing values
air_quality = air_quality.loc[
    (air_quality["C6H6(GT)"] != -200) &
    (air_quality["CO(GT)"] != -200) &
    (air_quality["T"] != -200) &
    (air_quality["RH"] != -200) &
    (air_quality["AH"] != -200)
]

air_quality.head()

,Date,Time,C6H6(GT),CO(GT),T,RH,AH
0,3/10/2004,18:00:00,11.9,2.6,13.6,48.9,0.7578
1,3/10/2004,19:00:00,9.4,2.0,13.3,47.7,0.7255
2,3/10/2004,20:00:00,9.0,2.2,11.9,54.0,0.7502
3,3/10/2004,21:00:00,9.2,2.2,11.0,60.0,0.7867
4,3/10/2004,22:00:00,6.5,1.6,11.2,59.6,0.7888


First we change the column date to a `pd.datetime` type object to ensure the dates are sorted in the proper way.\
Then it is grouped by day (Column `"Date"`).\
The average for each variable is calculated per day.

In [20]:

air_quality["Date"] = pd.to_datetime(air_quality["Date"])
air_quality = air_quality.groupby("Date")[["C6H6(GT)", "CO(GT)",
                                           "T", "RH", "AH"]].mean()
air_quality.head()

,C6H6(GT),CO(GT),T,RH,AH
Date,,,,,
2004-03-10,8.450000,1.966667,12.033333,54.900000,0.765633
2004-03-11,8.269565,2.239130,9.826087,64.230435,0.777039
2004-03-12,12.177273,2.804545,11.618182,50.190909,0.665164
2004-03-13,11.121739,2.695652,13.121739,50.682609,0.733013
2004-03-14,9.830435,2.469565,16.182609,48.317391,0.849209


It is added a helper column to give index for the days

In [21]:
air_quality["Day"] = range(1, len(air_quality) + 1)
air_quality.head()



,C6H6(GT),CO(GT),T,RH,AH,Day
Date,,,,,,
2004-03-10,8.450000,1.966667,12.033333,54.900000,0.765633,1
2004-03-11,8.269565,2.239130,9.826087,64.230435,0.777039,2
2004-03-12,12.177273,2.804545,11.618182,50.190909,0.665164,3
2004-03-13,11.121739,2.695652,13.121739,50.682609,0.733013,4
2004-03-14,9.830435,2.469565,16.182609,48.317391,0.849209,5


### Creating helper function to calculate the MSE per step

Defining a function that takes a dataframe with the predictors, a series with he response variable and the day until is going to be used to train the model, then this regression is going to be used to predict the next day and the mean squared error **(MSE)** is going to be calculated based on the predicted and the measured value.

In [22]:
import warnings

warnings.filterwarnings("ignore")  # hides all warnings from here on

# any code below will not show warnings


In [23]:
def MSE2(X , Y: pd.Series, Day: int) -> float:
    """
    This function calculates the mean squared error for a given day.
    Uses just the next day to calculate the MSE.
    X: is a dataframe with the predictors
    Y: is a series with the target variable
    Day: is the day until the mean squared error is calculated
    (including this day).
    """
    X_train = X.iloc[:Day] #slice the predictors until the especified date to work as training set
    Y_train = Y.iloc[:Day] # slice the response variable until the especified date
    X_test = X.iloc[Day] #get the predictors to test the model
    Y_test = [Y.iloc[Day]] # get the response value with which we are going to compared to the predicted value
    reg = sk.linear_model.LinearRegression()
    reg.fit(X_train.values, Y_train.values) #Fit a linear regression to the training values
    #print(reg.intercept_, reg.coef_)
    #Y_pred = reg.predict(X_test)
    if type(X_test) == pd.core.series.Series:
        Y_pred = reg.predict(X_test.to_numpy().reshape(1, -1))
    else:
        Y_pred = reg.predict(X_test.to_numpy())
    #MSE = (((Y_test - Y_pred)**2))
    MSE = sk.metrics.mean_squared_error(Y_test, Y_pred)
    return MSE
    #MSE(air_quality[["CO(GT)"]], air_quality["C6H6(GT)"],250) #Values to test the function
    #air_quality.iloc[249:251]
    #0.694715437209398 + 4.78962064 * 2.221739
    #((8.069565 - 11.336002408302358)**2)/250

Next we test the function with one predictor and obtain the MSE

In [24]:
MSE2(air_quality[["CO(GT)", "T"]], air_quality["C6H6(GT)"],250)

0.7161008160779453

In [25]:
#Testing value for the MSE2 function
X = air_quality[["CO(GT)", "T"]].iloc[250]
print(X)
print(air_quality["C6H6(GT)"].iloc[250])
Y = -3.6565302977532053 + 5.14640409 * X['CO(GT)'] + 0.17034599 * X['T']
print(Y)
((air_quality["C6H6(GT)"].iloc[250] - Y)**2)

CO(GT)    2.221739
T         6.682609
Name: 2004-12-22 00:00:00, dtype: float64
8.069565217391304
8.915792644072884


np.float64(0.7161008576681284)

### Construction of the function to calculate the CV error

In [26]:
def CV_error(X, Y, Day):
    M_total = 0
    for i in range(Day, len(Y)):
        #print(M_total)
        #print(MSE(X, Y, i))
        M_total+= MSE2(X, Y, i)
        #print(M_total)
    return M_total/(len(Y)-Day)

In [27]:
CV_error(air_quality[["CO(GT)"]], air_quality["C6H6(GT)"],250)

7.402897202768712

In [28]:
CV_error(air_quality[["CO(GT)", "T", "RH", "AH"]], air_quality["C6H6(GT)"],250)

5.0963115645839805

It is clear that based on the cross-validation, the multiple linear regression model (MLR) shows a lower MSE when predicting the amount of benzene in the air based on carbon monoxide concentration, temperaure, relative humidity and absolute humidity.

### Fitting the best model to the whole data

In [29]:
#This code chunk will fit the regression model that has the lowest MSE between the two evaluated
reg1 = sk.linear_model.LinearRegression()
reg1.fit(air_quality[["CO(GT)", "T", "RH", "AH"] ], air_quality["C6H6(GT)"]) #Fit a linear regression to the training values
print(reg1.intercept_, reg1.coef_) #

-1.8377694729981364 [ 4.77080433  0.11973259 -0.01620259  0.68866811]


The best model from the entire data set is:

$\widehat{Benzene}_{[\mu g/m^3]} = -1.838 + 4.771 * CO_{[mg/m^3]} + 0.120 * T_{°C} - 0.016 * RH_{\%} + 0.689 *AH$